In [2]:
# Import data

from datasets import load_dataset
ds = load_dataset("ucirvine/sms_spam")

In [3]:
import pandas as pd

df = pd.DataFrame(ds['train'])
print(df.head())

                                                 sms  label
0  Go until jurong point, crazy.. Available only ...      0
1                    Ok lar... Joking wif u oni...\n      0
2  Free entry in 2 a wkly comp to win FA Cup fina...      1
3  U dun say so early hor... U c already then say...      0
4  Nah I don't think he goes to usf, he lives aro...      0


In [8]:
# convert training dataframe to text file

df.to_csv('training_data.txt', sep='\t', index=False)

In [9]:
# Keep the original ham/spam ratio in both partitions.
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=1,
    stratify=df["label"],
)
train_df = train_df.copy()
test_df = test_df.copy()

print(f"Training messages: {len(train_df)}")
print(f"Testing messages:  {len(test_df)}")

Training messages: 4459
Testing messages:  1115


## Prepare the messages

This dataset encodes **ham as 0** and **spam as 1**. The tokenizer lowercases text, keeps words and numbers, and treats punctuation as a separator. We use the same tokenizer for the exploratory counts and the classifier.

In [10]:
import re
import numpy as np

label_name = {0: "ham", 1: "spam"}

# creates a compiled regular expression object that matches one or more alphanumeric words
# separated by single spaces, while ignoring underscores and punctuation
token_pattern = re.compile(r"[^\W_]+(?:['’][^\W_]+)*", flags=re.UNICODE) 

def tokenize(text):
    """Return normalized word and number tokens from one message."""
    if pd.isna(text):    # isna() detect the missing value
        return []
    return token_pattern.findall(str(text).casefold()) # casefold() use for lower all strings and unicode value

print(tokenize("I am ok"))

['i', 'am', 'ok']


In [11]:
unknown_labels = set(df["label"].dropna().unique()) - set(label_name)
if unknown_labels:
    raise ValueError(f"Unexpected dataset labels: {unknown_labels}")

train_df["tokens"] = train_df["sms"].map(tokenize) # map() -> Take each indivisual entry in column and pass through tokenize function
train_df[["sms", "tokens", "label"]].head()

,sms,tokens,label
2758,Yo we are watching a movie on netflix\n,"[yo, we, are, watching, a, movie, on, netflix]",0
595,I met you as a stranger and choose you as my f...,"[i, met, you, as, a, stranger, and, choose, yo...",0
4526,Cos i was out shopping wif darren jus now n i ...,"[cos, i, was, out, shopping, wif, darren, jus,...",0
4905,"no, i *didn't* mean to post it. I wrote it, an...","[no, i, didn't, mean, to, post, it, i, wrote, ...",0
3118,Now am free call me pa.\n,"[now, am, free, call, me, pa]",0


In [12]:
# Inspect the most frequent tokens in each class.
token_rows = train_df[["label", "tokens"]].explode("tokens").dropna(subset=["tokens"])
word_counts = (
    token_rows.groupby(["label", "tokens"], observed=True)
    .size()
    .rename("count")
    .reset_index()
)
top_words = (
    word_counts.sort_values(["label", "count"], ascending=[True, False])
    .groupby("label", group_keys=False)
    .head(10)
)
top_words.assign(class_name=top_words["label"].map(label_name))[
    ["class_name", "tokens", "count"]
]

,class_name,tokens,count
2625,ham,i,1859
6118,ham,you,1512
5463,ham,to,1264
5358,ham,the,905
5597,ham,u,826
156,ham,a,822
342,ham,and,678
2682,ham,in,661
3323,ham,me,644
3535,ham,my,608


In [13]:
# find probability of spam and ham that is P(spam) and P(ham) 

class_counts = train_df["label"].value_counts().sort_index()   # index: 0 = ham, 1 = spam
priors = class_counts / class_counts.sum()

p_ham = priors[0]
p_spam = priors[1]
print(f"P(ham)  = {p_ham:.4f}")
print(f"P(spam) = {p_spam:.4f}")

P(ham)  = 0.8659
P(spam) = 0.1341


In [17]:
# Find Word probabilities on spam and ham massege, i.e P(word | spam) & P(word | ham)
def word_probability(word_counts, k=0.5):
  
  count_table = (word_counts.pivot(index="tokens", columns="label", values="count").fillna(0))
  count_table.columns = ["ham_count", "spam_count"]       # 0 -> ham, 1 -> spam

  vocab_size = len(count_table)                            # V = number of distinct words

  total_ham_words = count_table["ham_count"].sum()
  total_spam_words = count_table["spam_count"].sum()

# P(word | class) = (count(word, class) + alpha) / (total words in class + alpha * V)
  count_table["P(word|ham)"] = (count_table["ham_count"] + k) / (total_ham_words + k * vocab_size)
  count_table["P(word|spam)"] = (count_table["spam_count"] + k) / (total_spam_words + k * vocab_size)

  return count_table.sort_values("spam_count", ascending=False)

print(word_probability(word_counts, k=0.5))

           ham_count  spam_count  P(word|ham)  P(word|spam)
tokens                                                     
to            1264.0       546.0     0.021284      0.028618
a              822.0       296.0     0.013844      0.015526
call           195.0       282.0     0.003291      0.014793
you           1512.0       241.0     0.025458      0.012646
your           345.0       218.0     0.005815      0.011442
...              ...         ...          ...           ...
dangerous        1.0         0.0     0.000025      0.000026
ü              138.0         0.0     0.002331      0.000026
ü'll             3.0         0.0     0.000059      0.000026
〨ud              1.0         0.0     0.000025      0.000026
zealand          1.0         0.0     0.000025      0.000026

[7791 rows x 4 columns]


In [23]:
# Find P(spam | word) and P (ham | word)
word_probs = word_probability(word_counts, k=0.5)

def spam_probability(message, k=0.5):
  message_word = tokenize(message)

  if not message_word:
        return pd.DataFrame(columns=["in_vocab", "P(word|spam)", "P(word|ham)",
                                     "P(spam|word)", "P(ham|word)"])
  
  word_rows = word_probs.reindex(message_word).copy()
  word_rows["in_vocab"] = word_rows["P(word|spam)"].notna()

  vocab_size = len(word_probs)
  total_ham_words = word_probs["ham_count"].sum()
  total_spam_words = word_probs["spam_count"].sum()

  word_rows["P(word|spam)"] = word_rows["P(word|spam)"].fillna(
      k / (total_spam_words + k * vocab_size)
  )
  word_rows["P(word|ham)"] = word_rows["P(word|ham)"].fillna(
      k / (total_ham_words + k * vocab_size)
  )
  
  spam_weight = word_rows["P(word|spam)"] * p_spam
  ham_weight = word_rows["P(word|ham)"] * p_ham
  denominator = spam_weight + ham_weight

  word_rows["P(spam|word)"] = spam_weight / denominator
  word_rows["P(ham|word)"] = ham_weight / denominator
  return word_rows[["in_vocab", "P(word|spam)", "P(word|ham)",
                    "P(spam|word)", "P(ham|word)"]]
  
print(spam_probability("free zorblax call now"))

         in_vocab  P(word|spam)  P(word|ham)  P(spam|word)  P(ham|word)
tokens                                                                 
free         True      0.009400     0.000850      0.631365     0.368635
zorblax     False      0.000026     0.000008      0.325167     0.674833
call         True      0.014793     0.003291      0.410474     0.589526
now          True      0.008876     0.004183      0.247365     0.752635


In [25]:
def classify_message(message):
    rows = spam_probability(message, k=0.5)
    if rows.empty:
        return {"P(spam)": p_spam, "P(ham)": p_ham}

    log_spam = np.log(p_spam) + np.log(rows["P(word|spam)"]).sum()
    log_ham  = np.log(p_ham)  + np.log(rows["P(word|ham)"]).sum()

    # normalize in a numerically stable way
    m = max(log_spam, log_ham)
    p_s = np.exp(log_spam - m) / (np.exp(log_spam - m) + np.exp(log_ham - m))
    return {"P(spam)": p_s, "P(ham)": 1 - p_s}

classify_message("free entry call now to win zorblax")

{'P(spam)': np.float64(0.9999940561569367),
 'P(ham)': np.float64(5.943843063294096e-06)}

In [37]:
# Create class for Naive Bayes Classifier

class NaiveBayes_Classifier:
        
    def __init__(self, k=0.5):
        self.k = k
        self.word_probs = None
        self.p_ham = None
        self.p_spam = None

    def train(self, train_df):
        # Count spam and ham massages
        class_counts = train_df["label"].value_counts().reindex([0, 1], fill_value=0)
        self.p_ham = class_counts[0] / class_counts.sum()
        self.p_spam = class_counts[1] / class_counts.sum()

        token_rows = train_df[["label", "sms"]].copy()
        token_rows["tokens"] = token_rows["sms"].map(tokenize)
        token_rows = token_rows[["label", "tokens"]].explode("tokens").dropna(subset=["tokens"])
        word_counts = (token_rows.groupby(["label", "tokens"], observed=True).size().rename("count").reset_index())
        self.word_probs = word_probability(word_counts, k=self.k)

    def classify(self, message):
        if self.word_probs is None:
            raise RuntimeError("Train the classifier before calling classify.")

        message_tokens = tokenize(message)
        if not message_tokens:
            return {"P(spam)": self.p_spam, "P(ham)": self.p_ham}

        rows = self.word_probs.reindex(message_tokens)
        vocab_size = len(self.word_probs)
        total_ham_words = self.word_probs["ham_count"].sum()
        total_spam_words = self.word_probs["spam_count"].sum()

        rows["P(word|spam)"] = rows["P(word|spam)"].fillna(
            self.k / (total_spam_words + self.k * vocab_size)
        )
        rows["P(word|ham)"] = rows["P(word|ham)"].fillna(
            self.k / (total_ham_words + self.k * vocab_size)
        )

        log_spam = np.log(self.p_spam) + np.log(rows["P(word|spam)"]).sum()
        log_ham = np.log(self.p_ham) + np.log(rows["P(word|ham)"]).sum()

        # normalize in a numerically stable way
        m = max(log_spam, log_ham)
        p_s = np.exp(log_spam - m) / (np.exp(log_spam - m) + np.exp(log_ham - m))

        return {"P(spam)": p_s, "P(ham)": 1 - p_s}

    def predict(self, test_df, threshold=0.5):
        
        return test_df["sms"].apply(
            lambda message: int(self.classify(message)["P(spam)"] >= threshold)
        )


In [38]:
nb = NaiveBayes_Classifier()
nb.train(train_df)
prediction = nb.predict(test_df)
print(prediction)

78      0
2487    0
444     0
2113    1
2115    1
       ..
1872    0
3287    0
5208    0
2321    0
2135    0
Name: sms, Length: 1115, dtype: int64


In [39]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score

print(f"Confusion matrix of data frame for test data:\n{confusion_matrix(test_df["label"],prediction)}")
print(f"Accuracy score for test data: {accuracy_score(test_df["label"], prediction)}")
print(f"Precision score for test data: {precision_score(test_df["label"], prediction)}")


Confusion matrix of data frame for test data:
[[947  19]
 [  5 144]]
Accuracy score for test data: 0.97847533632287
Precision score for test data: 0.8834355828220859
